# E2: optimización de implementación con once orígenes

Se conserva el calendario original: enero a octubre de 2024 y enero de 2025, común para los 14 sistemas neuronales. La reducción a tres orígenes no es operativa hasta justificar su suficiencia. Se mantienen seis familias NAS, 1.000 arquitecturas por familia/origen y el máximo de 85.074 intentos de selección. No se recortan otros requisitos de E2 ni se declara terminado el benchmark.

La implementación selectiva de TimesNet se mide en la GPU real y sólo se habilita en casos que pasan las comprobaciones numéricas y mejoran al menos 5 %. Los demás usan el cálculo original. Se priorizan los modelos con cuerpo fijo y se reutilizan resultados compatibles sin contarlos como nuevos ajustes.

**Espera el cierre del controlador anterior; no ejecutes ambos.** En configuración fija RUN_SEARCH=True y conserva RETRY_LOST_ONCE=False. Ejecuta 1–7. En la sección 8 activa RUN_OPTIMIZED_SELECTION=True una sola vez. El secreto GITHUB_TOKEN debe tener acceso para este notebook.

Por defecto se transfiere sólo el saldo del límite original de 48 horas, sin reinicio. Si pausa por presupuesto, la sección 11 permite registrar una ampliación manual mediante un NUEVO LÍMITE TOTAL introducido expresamente por el usuario. Publicar o abrir este notebook no amplía el presupuesto automáticamente. El saldo puede no alcanzar para todas las corridas. Si existe una campaña de optimización anterior, se bloquea la nueva hasta conciliar su saldo y caché. No se repiten automáticamente intentos sin resultado.


In [ ]:
from pathlib import Path
import os, re, sys, json, hashlib, subprocess, platform, shutil, math

SOURCE_COMMIT = "564ba4762f84a800b02fa4c734f5752a169f5223"
EXPECTED_SEARCH_REGISTRATION_SHA256 = "78c3c5b8ca042a347aaa80d642f3affd8b5911e8697d8747a2de915d1c3194a9"
DATA_COMMIT = "0b079b50625a936456a70695532853fc13298f31"
REPOSITORY_URL = "https://github.com/jassan-bit/DeepLearning_202630.git"

RUN_SEARCH = False
RETRY_LOST_ONCE = False
SHOW_PLAN = True
MODELS = None  # None conserva los catorce sistemas neuronales del plan.
ORIGINS = None  # None conserva los once orígenes de desarrollo admitidos.
DEVICE = "cuda:0"
SESSION_HOURS = 8
CAMPAIGN_HOURS = 24
EXTEND_CAMPAIGN_HOURS = None  # Ampliación explícita registrada al continuar; conserva el límite inicial anterior.
STORAGE_GIB = 10
CAMPAIGN_NAME = "e2-cuda-selection-v1"
GITHUB_REPOSITORY = "jassan-bit/DeepLearning_202630"
GITHUB_BRANCH_PREFIX = "campaigns"
GITHUB_PATH_PREFIX = "campaigns"

# Reutiliza el entorno y la caché del piloto; mantiene separado el código nuevo.
WORKSPACE = Path.cwd()/"entregable2_gpu_workspace"
REPO = WORKSPACE/"search-source"
VENV = WORKSPACE/"venv-py312-cu126"
DATA_CACHE = WORKSPACE/"development_cache"
LOCAL_CAMPAIGN_CACHE = WORKSPACE/"search-campaigns"
UV_VERSION = "0.9.5"
MANAGED_PYTHON_VERSION = "3.12.12"
PIP_VERSION = "25.3"
UV_WHEEL_SHA256 = "6507bbbcd788553ec4ad5a96fa19364dc0f58b023e31d79868773559a83ec181"
PIP_WHEEL_SHA256 = "9655943313a94722b7774661c21049070f6bbb0a1516bf02f7c8d5d9201514cd"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CUDA_ENV = dict(os.environ)
CUDA_ENV.update(CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONNOUSERSITE="1",
                OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
CUDA_ENV.pop("PYTHONPATH", None)
CUDA_ENV.pop("GITHUB_TOKEN", None)
CUDA_ENV.pop("GH_TOKEN", None)
CUDA_ENV.update(UV_CACHE_DIR=str(WORKSPACE/"uv-cache"),
                UV_PYTHON_INSTALL_DIR=str(WORKSPACE/"managed-python"),
                UV_PYTHON_BIN_DIR=str(WORKSPACE/"managed-python-bin"), UV_NATIVE_TLS="true")
SOURCE_READY = False
PLAN_READY = False
REMOTE_READY = False
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
SEARCH_PLAN = None
EXECUTION_ATTEMPTED = False
LAST_SEARCH_RETURN_CODE = None
CAMPAIGN_OUTPUT = None
GITHUB_ENV = None

def run(args, *, cwd=None, capture=False, input_text=None):
    return subprocess.run([str(x) for x in args], cwd=cwd, env=CUDA_ENV, check=True,
                          text=True, input=input_text, capture_output=capture)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()


## 1. Código inmutable, sin checkout de observaciones

El modo de planificación funciona sin GPU. El checkout contiene código, pruebas y protocolos; no incluye los CSV ni carpetas de resultados. Una fuente diferente exige otro checkout y no modifica la campaña existente.


In [ ]:
SPARSE_PATHS = [
    "/scripts/", "/tests/", "/requirements-e2-training-gpu.lock.txt",
    "/book_e2/_static/search_protocol_20261109/protocol.json",
    "/experiments/dl_r2_cv_20261104/protocol.json",
    "/experiments/e2_development_pilot_20261109/source_protocol.json",
    "/experiments/e2_gpu_pilot_20261109/", "/experiments/e2_cuda_search_20261109/",
]
SOURCE_READY = False
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise RuntimeError("Falta el commit publicado de 40 caracteres; no usar main ni un marcador.")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("El entorno fijado corresponde a Linux x86_64.")
if not REPO.exists():
    REPO.mkdir(parents=True)
    run(["git", "init", REPO])
    run(["git", "-C", REPO, "remote", "add", "origin", REPOSITORY_URL])
    run(["git", "-C", REPO, "config", "core.sparseCheckout", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.promisor", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.partialclonefilter", "blob:none"])
    run(["git", "-C", REPO, "sparse-checkout", "set", "--no-cone", "--stdin"],
        input_text="\n".join(SPARSE_PATHS)+"\n")
    run(["git", "-C", REPO, "fetch", "--filter=blob:none", "--depth=1", "origin", SOURCE_COMMIT])
    run(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"])
if run(["git", "-C", REPO, "rev-parse", "HEAD"], capture=True).stdout.strip() != SOURCE_COMMIT:
    raise RuntimeError("El checkout existente pertenece a otro commit; usa un WORKSPACE nuevo.")
if (REPO/"results").exists():
    raise RuntimeError("Este checkout debe contener sólo código; usa una carpeta nueva sin results.")
SOURCE_READY = True
print("Fuente verificada:", SOURCE_COMMIT)


## 2. Mostrar el plan antes de la GPU o los datos

Los límites y el dominio quedan identificados antes del acceso al desarrollo. El modo `--plan-only` no crea una campaña, no abre los CSV ni importa CUDA. El límite de sesión puede dejar trabajos pendientes; el límite global no equivale a una estimación del experimento completo. Los límites científicos por etapa permanecen separados de estos techos operativos.

`ORIGINS=None` conserva los once cortes de desarrollo y `MODELS=None` los catorce sistemas neuronales. Los subconjuntos se identifican como cobertura parcial; no se presentan como todo el benchmark. Para una ampliación global posterior, conserva `CAMPAIGN_HOURS=24` y fija explícitamente `EXTEND_CAMPAIGN_HOURS`; el ejecutor registra la enmienda sin cambiar candidatos ni semillas.


In [ ]:
PLAN_READY = False
SEARCH_PLAN = None
if not SOURCE_READY:
    raise RuntimeError("Verifica primero la fuente congelada.")
if not re.fullmatch(r"[A-Za-z0-9_.-]{1,80}", CAMPAIGN_NAME) or CAMPAIGN_NAME in {".", ".."}:
    raise ValueError("Elige un nombre simple de campaña, sin rutas ni espacios.")
for value, label in [(SESSION_HOURS, "SESSION_HOURS"), (CAMPAIGN_HOURS, "CAMPAIGN_HOURS"), (STORAGE_GIB, "STORAGE_GIB")]:
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
        raise ValueError(label+" debe ser un número finito positivo.")
if SESSION_HOURS > CAMPAIGN_HOURS:
    raise ValueError("El límite de sesión no puede superar el límite global.")
if EXTEND_CAMPAIGN_HOURS is not None:
    if isinstance(EXTEND_CAMPAIGN_HOURS, bool) or not isinstance(EXTEND_CAMPAIGN_HOURS, (int, float)) or not math.isfinite(EXTEND_CAMPAIGN_HOURS) or EXTEND_CAMPAIGN_HOURS <= CAMPAIGN_HOURS:
        raise ValueError("La ampliación explícita debe superar el límite global inicial.")
CAMPAIGN_OUTPUT = LOCAL_CAMPAIGN_CACHE/CAMPAIGN_NAME
GITHUB_BRANCH = GITHUB_BRANCH_PREFIX+"/"+CAMPAIGN_NAME
GITHUB_PREFIX = GITHUB_PATH_PREFIX+"/"+CAMPAIGN_NAME
PROTOCOL_RELATIVE = "book_e2/_static/search_protocol_20261109/protocol.json"
NORMALIZED_PROTOCOL_SHA256 = "e0030c990a4afa15665aec254211b0d9e680e9ac5d92dfb8094e43e965152abe"

def verify_source_snapshot(*, check_protocol=True):
    changed = run(["git", "-C", REPO, "diff", "--name-only", "HEAD"], capture=True).stdout.splitlines()
    if any(name != PROTOCOL_RELATIVE for name in changed):
        raise RuntimeError("Hay cambios en la fuente congelada; conserva la campaña y revisa el checkout.")
    untracked = run(["git", "-C", REPO, "ls-files", "--others", "--exclude-standard", "scripts"], capture=True).stdout.splitlines()
    if any(name.endswith(".py") for name in untracked):
        raise RuntimeError("Hay módulos Python no registrados en scripts.")
    if check_protocol and sha256_file(REPO/PROTOCOL_RELATIVE) != NORMALIZED_PROTOCOL_SHA256:
        raise RuntimeError("El protocolo cambió respecto de los bytes registrados.")

verify_source_snapshot(check_protocol=False)
run([sys.executable, REPO/"scripts/prepare_e2_pilot_snapshot.py", "--root", REPO], cwd=REPO)
if sha256_file(REPO/PROTOCOL_RELATIVE) != NORMALIZED_PROTOCOL_SHA256:
    raise RuntimeError("El protocolo no coincide con los bytes registrados.")

def search_command(python, *, execute):
    command = [python, REPO/"scripts/run_e2_cuda_search.py", "--execute" if execute else "--plan-only",
               "--source-commit", SOURCE_COMMIT, "--expected-registration-sha256", EXPECTED_SEARCH_REGISTRATION_SHA256,
               "--output", CAMPAIGN_OUTPUT, "--session-hours", SESSION_HOURS,
               "--campaign-hours", CAMPAIGN_HOURS, "--storage-gib", STORAGE_GIB,
               "--device", DEVICE,
               "--github-repository", GITHUB_REPOSITORY, "--github-branch", GITHUB_BRANCH,
               "--github-prefix", GITHUB_PREFIX]
    if MODELS is not None:
        command += ["--models", *MODELS]
    if ORIGINS is not None:
        command += ["--origins", *ORIGINS]
    if execute:
        if EXTEND_CAMPAIGN_HOURS is not None:
            command += ["--extend-campaign-hours", EXTEND_CAMPAIGN_HOURS]
        command += ["--data-root", DATA_CACHE,
                    "--source-root-itransformer", REPO/"work/author_sources/iTransformer"]
    return command

if SHOW_PLAN or RUN_SEARCH:
    planned = run(search_command(sys.executable, execute=False), cwd=REPO, capture=True)
    SEARCH_PLAN = json.loads(planned.stdout)
    print("Identidad histórica A3 verificada. El calendario original de once orígenes se ejecuta en la sección 8.")
    PLAN_READY = True
    if EXTEND_CAMPAIGN_HOURS is not None:
        print("Ampliación global solicitada al ejecutar:", EXTEND_CAMPAIGN_HOURS, "horas. Se registrará como enmienda.")
else:
    print("Activa SHOW_PLAN para revisar el dominio antes del lanzamiento.")


## 3. Configurar el guardado en GitHub

La campaña se guarda en una rama dedicada de **`jassan-bit/DeepLearning_202630`**. El disco de Colab es sólo una caché recuperable. El modo de planificación no usa credenciales ni escribe en el repositorio.

Para ejecutar desde Colab, crea un [token personal de GitHub con alcance fino](https://github.com/settings/personal-access-tokens/new) con acceso al repositorio elegido y permiso **Contents: read and write**. En el panel **Secretos** de Colab, añade `GITHUB_TOKEN` y habilita su acceso para este notebook. También puede recibirse como variable de entorno en un servidor propio. No pegues el token en una celda, un archivo de configuración o un mensaje.

La lectura del código público no necesita ese token. Sólo la sincronización de la campaña lo utiliza. Se verifica el acceso al repositorio; el ejecutor exige además confirmar en GitHub el estado de inicio antes de entrenar y el resultado antes de avanzar al siguiente trabajo. Un error de sincronización pausa la campaña. Los JSON autorizados no contienen credenciales, rutas internas ni pesos de modelos.


In [ ]:
REMOTE_READY = False
GITHUB_ENV = None

def github_execution_environment():
    import urllib.request, urllib.error
    if GITHUB_REPOSITORY != "jassan-bit/DeepLearning_202630":
        raise RuntimeError("Este notebook fija el repositorio autorizado para la campaña.")
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN").strip()
        except Exception:
            raise RuntimeError("Añade GITHUB_TOKEN en Secretos de Colab y habilita su acceso al notebook.") from None
    if not token or any(character.isspace() for character in token):
        raise RuntimeError("El secreto GITHUB_TOKEN está vacío o tiene un formato inválido.")
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, request, response, code, message, headers, new_url):
            return None
    request = urllib.request.Request("https://api.github.com/repos/"+GITHUB_REPOSITORY,
        headers={"Authorization":"Bearer "+token, "Accept":"application/vnd.github+json",
                 "X-GitHub-Api-Version":"2022-11-28"}, method="GET")
    try:
        with urllib.request.build_opener(NoRedirect()).open(request, timeout=30) as response:
            raw = response.read(65537)
            if len(raw) > 65536:
                raise ValueError("Unexpected response size")
            payload = json.loads(raw)
    except urllib.error.HTTPError as error:
        raise RuntimeError("GitHub rechazó la comprobación de acceso (HTTP "+str(error.code)+"). Revisa el secreto y su repositorio autorizado.") from None
    except Exception:
        raise RuntimeError("No se pudo comprobar el acceso a GitHub; no se habilitó el entrenamiento.") from None
    if not isinstance(payload, dict) or payload.get("full_name", "").lower() != GITHUB_REPOSITORY.lower():
        raise RuntimeError("GitHub no confirmó el repositorio esperado.")
    environment = dict(CUDA_ENV)
    environment["GITHUB_TOKEN"] = token
    return environment

if RUN_SEARCH:
    if not PLAN_READY:
        raise RuntimeError("Revisa primero el plan sin entrenamiento.")
    GITHUB_ENV = github_execution_environment()
    REMOTE_READY = True
    print("Acceso al repositorio confirmado. El ejecutor comprobará el guardado remoto antes de cada ajuste.")
else:
    print("Planificación: no se leyeron secretos ni se escribió en GitHub.")


## 4. Admitir una GPU real antes de instalar CUDA

Activa GPU en Colab antes de `RUN_SEARCH=True`. Una biblioteca CUDA instalada no basta. Esta sección comprueba Linux x86_64 y una tarjeta NVIDIA; el entorno aislado comprobará después una operación CUDA real. No hay sustitución automática por CPU.


In [ ]:
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
if RUN_SEARCH:
    if not REMOTE_READY or not PLAN_READY:
        raise RuntimeError("Verifica el plan y el guardado en GitHub antes de instalar CUDA.")
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
        raise RuntimeError("SOURCE_COMMIT debe contener el SHA GPU publicado de 40 caracteres; no usar main.")
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("El lock CUDA corresponde a Linux x86_64.")
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("No hay GPU NVIDIA visible. Activa GPU en el proveedor antes de continuar.")
    probe = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture=True)
    if not probe.stdout.strip():
        raise RuntimeError("nvidia-smi no informó ninguna GPU.")
    print(probe.stdout.strip())
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    HARDWARE_READY = True
    print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))
else:
    print("Planificación: no se consultó ni inicializó una GPU.")


## 5. Reutilizar el Python aislado y comprobar CUDA

Se reutiliza el entorno del piloto si sus versiones coinciden. El bootstrap fija uv 0.9.5, CPython 3.12.12 y pip 25.3, comprueba los wheels por SHA256 e instala el lock PyTorch 2.8.0 + CUDA 12.6. No modifica paquetes del kernel anfitrión. La instalación y las pruebas previas no forman parte del cronómetro del ejecutor; tampoco definen el tiempo cobrado o la cuota de GPU del proveedor.


In [ ]:
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
if RUN_SEARCH:
    if not HARDWARE_READY:
        raise RuntimeError("Comprueba primero Linux y la GPU NVIDIA en la sección 1.")
    
    def bootstrap_python312():
        bootstrap = WORKSPACE/"bootstrap"
        bootstrap.mkdir(parents=True, exist_ok=True)
        uv_home = bootstrap/("uv-"+UV_VERSION)
        uv_binary = uv_home/"bin/uv"
        uv_requirements = bootstrap/"uv.requirements.txt"
        uv_requirements.write_text("uv=="+UV_VERSION+" --hash=sha256:"+UV_WHEEL_SHA256+"\n", encoding="utf-8")
        if not uv_binary.is_file():
            if uv_home.exists():
                raise RuntimeError("El bootstrap uv está incompleto; elige un WORKSPACE nuevo.")
            run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir",
                 "--no-deps", "--only-binary=:all:", "--require-hashes", "--index-url", "https://pypi.org/simple",
                 "--target", uv_home, "-r", uv_requirements])
        uv_version = run([uv_binary, "--version"], capture=True).stdout.strip().split()
        if uv_version[:2] != ["uv", UV_VERSION]:
            raise RuntimeError("La versión local de uv no coincide con la fijada.")
        managed_directory = WORKSPACE/"managed-python"
        run([uv_binary, "python", "install", MANAGED_PYTHON_VERSION,
             "--install-dir", managed_directory, "--no-bin", "--no-config"])
        managed_python = Path(run([uv_binary, "python", "find", MANAGED_PYTHON_VERSION,
                                  "--managed-python", "--no-python-downloads", "--no-project", "--no-config"],
                                 capture=True).stdout.strip())
        if managed_directory.resolve() not in managed_python.resolve().parents:
            raise RuntimeError("uv seleccionó un Python fuera del directorio gestionado.")
        if not (VENV/"bin/python").is_file():
            run([uv_binary, "venv", "--python", managed_python, "--no-python-downloads", "--no-project",
                 "--no-config", VENV])
        python = VENV/"bin/python"
        info = json.loads(run([python, "-c", "import sys,json; print(json.dumps({'version':list(sys.version_info[:3])}))"],
                              capture=True).stdout)
        if info["version"] != [3, 12, 12]:
            raise RuntimeError("El venv no usa CPython 3.12.12; elige una carpeta nueva.")
        pip_requirements = bootstrap/"pip.requirements.txt"
        pip_requirements.write_text("pip=="+PIP_VERSION+" --hash=sha256:"+PIP_WHEEL_SHA256+"\n", encoding="utf-8")
        run([uv_binary, "pip", "install", "--python", python, "--no-deps", "--only-binary=:all:",
             "--require-hashes", "--index-url", "https://pypi.org/simple", "--no-config", "-r", pip_requirements])
        run([python, "-c", "import pip; assert pip.__version__=='25.3'"])
        print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))
        print("Python aislado del buscador:", ".".join(map(str, info["version"])), "con pip", PIP_VERSION)
        return python
    
    GPU_PYTHON = bootstrap_python312()
    run([GPU_PYTHON, "-m", "pip", "install", "-r", REPO/"requirements-e2-training-gpu.lock.txt"])
    run([GPU_PYTHON, "-m", "pip", "check"])
    run([GPU_PYTHON, "-c", "import sys,torch; assert sys.version_info[:3]==(3,12,12); assert torch.__version__=='2.8.0+cu126'; assert torch.version.cuda=='12.6'; print('Python/Torch/CUDA: versiones verificadas')"])
    # Sólo la prueba CUDA real habilita las siguientes celdas; no se abren datos aquí.
    run([GPU_PYTHON, REPO/"scripts/run_e2_gpu_pilot.py", "--check-environment", "--device", DEVICE], cwd=REPO)
    GPU_READY = True
else:
    print("Planificación: no se instalaron paquetes ni se ejecutaron kernels CUDA.")


## 6. Fuentes y pruebas artificiales antes de los CSV

Se verifican las ocho fuentes oficiales de iTransformer. Una caché válida puede reutilizarse; los archivos o recibos diferentes se conservan y bloquean el lanzamiento. Las pruebas artificiales de CUDA y del adaptador se ejecutan antes de descargar observaciones. Sus ajustes sintéticos no son resultados HPO/NAS sobre mercado.


In [ ]:
SYNTHETIC_READY = False
DATA_READY = False
def ensure_itransformer_sources():
    import ast
    source = REPO/"scripts/fetch_e2_itransformer_sources.py"
    output = REPO/"work/author_sources/iTransformer"
    def reject_symlinks(path):
        if any(part.is_symlink() for part in (path.absolute(), *path.absolute().parents)):
            raise RuntimeError("El árbol iTransformer no admite enlaces simbólicos.")
    reject_symlinks(source)
    reject_symlinks(output)
    frozen = run(["git", "-C", REPO, "show", SOURCE_COMMIT+":scripts/fetch_e2_itransformer_sources.py"], capture=True).stdout
    if source.read_bytes() != frozen.encode():
        raise RuntimeError("El descargador iTransformer difiere de la fuente congelada.")
    constants = {node.targets[0].id: ast.literal_eval(node.value)
                 for node in ast.parse(frozen).body if isinstance(node, ast.Assign)
                 and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name)
                 and node.targets[0].id == "MANIFEST"}
    manifest, maximum = constants["MANIFEST"], 1024*1024
    if not output.exists():
        run([GPU_PYTHON, source, "--output", "work/author_sources/iTransformer"], cwd=REPO)
    if not output.is_dir():
        raise RuntimeError("El destino iTransformer no es un directorio válido.")
    for path in output.rglob("*"):
        reject_symlinks(path)
        relative = path.relative_to(output).as_posix()
        if path.is_file() and path.suffix in {".py", ".pyc", ".pyo", ".pth", ".so"} and relative not in manifest["files"]:
            if "__pycache__" not in path.relative_to(output).parts or path.suffix != ".pyc":
                raise RuntimeError("Hay código no registrado en el árbol iTransformer.")
    for relative, expected in manifest["files"].items():
        path = output/relative
        reject_symlinks(path)
        if not path.is_file() or path.stat().st_size > maximum or sha256_file(path) != expected:
            raise RuntimeError("Fuente iTransformer ausente o con hash distinto: "+relative)
    receipt = output/"source_receipt.json"
    reject_symlinks(receipt)
    expected_receipt = {"status":"complete", "repository":manifest["repository"], "commit":manifest["commit"],
                        "verified_files":manifest["files"], "weights_downloaded":False, "market_data_read":False}
    if not receipt.is_file() or receipt.stat().st_size > maximum or json.loads(receipt.read_text()) != expected_receipt:
        raise RuntimeError("El recibo iTransformer no coincide con los ocho archivos registrados.")
    print("iTransformer: ocho fuentes verificadas; caché reutilizable.")

if RUN_SEARCH:
    if not GPU_READY or not REMOTE_READY:
        raise RuntimeError("Comprueba CUDA y el acceso remoto antes de los tests.")
    ensure_itransformer_sources()
    smoke_env = dict(CUDA_ENV)
    smoke_env["E2_ITRANSFORMER_SOURCE_ROOT"] = str(REPO/"work/author_sources/iTransformer")
    for pattern in ["test_e2_cuda_models.py", "test_e2_cuda_training.py", "test_e2_cuda_bindings.py", "test_e2_search_session.py"]:
        subprocess.run([str(GPU_PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
                       cwd=REPO, env=smoke_env, check=True)
    SYNTHETIC_READY = True
    print("Comprobaciones artificiales completadas antes del acceso al desarrollo.")
else:
    print("Planificación: no se descargaron fuentes de autor ni se ajustaron modelos sintéticos.")


## 7. Obtener únicamente el desarrollo registrado

Con la ejecución desactivada no se descargan CSV. Al activarla se admiten sólo los derivados de 2023–2024 y su protocolo fijados por hash. El helper puede reutilizar la caché existente y rechaza bytes distintos. La selección recorre los once orígenes declarados en el plan, o el subconjunto explícito de `ORIGINS`. El origen 2025-01-01 es un marcador de calendario que utiliza sólo desarrollo disponible hasta 2024; sus valores finales no se abren.


In [ ]:
DATA_READY = False
DATA_FILES = {
    "results/eda_development/daily_market_development.csv": "bf6bb5160af924aef6a15c3684aeae932df3017e8639888bb7e91ce787900171",
    "results/eda_development/daily_volatility_development.csv": "20b96374a659eb87dadfa68aa3d6b6b5fb6bea5b10eebe6f2c186ad69861844a",
}
if RUN_SEARCH:
    if not GPU_READY or not PLAN_READY or not REMOTE_READY or not SYNTHETIC_READY:
        raise RuntimeError("Verifica primero el plan, CUDA, las pruebas artificiales y el guardado en GitHub.")
    run([GPU_PYTHON, REPO/"scripts/fetch_e2_development_sources.py", "--output", DATA_CACHE], cwd=REPO)
    for relative, checksum in DATA_FILES.items():
        if sha256_file(DATA_CACHE/relative) != checksum:
            raise RuntimeError("El derivado no coincide con el SHA256 registrado.")
    DATA_READY = True
    print("Desarrollo verificado; no se descargaron valores finales.")
else:
    print("Planificación: no se descargaron los CSV.")


## 8. Perfil CUDA y selección con cobertura prioritaria

La primera admisión mide 15 casos sintéticos en la GPU, antes de abrir desarrollo para nuevos ajustes. Los casos sin mejora conservan el cálculo original. Luego valida la caché histórica, sella la implementación y transfiere sólo el saldo. Si no hay saldo o hay un intento sin resultado, no continúa a ciegas.
Corrección técnica: el perfil medido se convierte a tipos JSON básicos antes de crear el diario. Resuelve el rechazo de `TorchVersion` sin cambiar valores, hash del perfil, cálculos, arquitecturas ni saldo. Se usa un checkout de controlador separado; el anterior no se modifica.


In [ ]:
RUN_OPTIMIZED_SELECTION = False
OPTIMIZATION_COMMIT = "8e3ab6cfb1b3fc3c804f0c914fa5d92b661058d9"
OPTIMIZATION_SOURCE = WORKSPACE/"optimization-source-11-v2-json-metadata"
OPTIMIZATION_OUTPUT = LOCAL_CAMPAIGN_CACHE/"e2-cuda-crop-11-v2"

if not RUN_OPTIMIZED_SELECTION:
    print("Preparación: máximo 85.074 intentos; conserva las seis familias NAS y el saldo previo.")
else:
    import signal
    if not RUN_SEARCH or not all([SOURCE_READY, PLAN_READY, REMOTE_READY, GPU_READY, SYNTHETIC_READY, DATA_READY]):
        raise RuntimeError("Completa 1–7 con RUN_SEARCH=True antes de ejecutar.")
    if RETRY_LOST_ONCE:
        raise RuntimeError("Conserva RETRY_LOST_ONCE=False: no se autoriza otra repetición R3.")
    for prefix in ("E2_RECOVERY_V4", "E2_RECOVERY_V3", "E2_GUIDE", "E2_OPT"):
        active = globals().get(prefix+"_ACTIVE_PROCESS")
        if active is not None and active.poll() is None:
            raise RuntimeError("Hay un controlador activo. Espera su cierre.")
    markers = ("run_e2_cuda_search.py", "run_e2_cuda_recovery_", "e2_guide_schedule.py", "run_e2_optimized_selection")
    processes = subprocess.run(["ps", "-eo", "pid=,args="], check=True, capture_output=True, text=True).stdout
    for row in processes.splitlines():
        fields = row.strip().split(maxsplit=1)
        if len(fields)==2 and any(marker in fields[1] for marker in markers):
            raise RuntimeError("Se encontró otro proceso de búsqueda; no se lanzó otra campaña.")
    verify_source_snapshot()
    paths = ["/scripts/e2_timesnet_exact_crop.py", "/scripts/run_e2_optimized_selection_v2.py",
             "/tests/test_e2_implementation_optimization_v2.py",
             "/experiments/e2_implementation_optimization_11_origins_20261111/registration_json_metadata_fix.json"]
    if not OPTIMIZATION_SOURCE.exists():
        OPTIMIZATION_SOURCE.mkdir(parents=True)
        run(["git", "init", OPTIMIZATION_SOURCE])
        run(["git", "-C", OPTIMIZATION_SOURCE, "remote", "add", "origin", REPOSITORY_URL])
        run(["git", "-C", OPTIMIZATION_SOURCE, "config", "core.sparseCheckout", "true"])
        run(["git", "-C", OPTIMIZATION_SOURCE, "config", "remote.origin.promisor", "true"])
        run(["git", "-C", OPTIMIZATION_SOURCE, "config", "remote.origin.partialclonefilter", "blob:none"])
        run(["git", "-C", OPTIMIZATION_SOURCE, "sparse-checkout", "set", "--no-cone", "--stdin"],input_text="\n".join(paths)+"\n")
        run(["git", "-C", OPTIMIZATION_SOURCE, "fetch", "--filter=blob:none", "--depth=1", "origin", OPTIMIZATION_COMMIT])
        run(["git", "-C", OPTIMIZATION_SOURCE, "checkout", "--detach", "FETCH_HEAD"])
    if run(["git", "-C", OPTIMIZATION_SOURCE, "rev-parse", "HEAD"],capture=True).stdout.strip()!=OPTIMIZATION_COMMIT:
        raise RuntimeError("El checkout pertenece a otra versión; no se modificó.")
    test_env = dict(CUDA_ENV)
    test_env["PYTHONPATH"] = os.pathsep.join([str(OPTIMIZATION_SOURCE/"scripts"),str(REPO/"scripts")])
    subprocess.run([str(GPU_PYTHON), "tests/test_e2_implementation_optimization_v2.py"],
                   cwd=OPTIMIZATION_SOURCE,env=test_env,check=True)
    command = [str(GPU_PYTHON),"-u",str(OPTIMIZATION_SOURCE/"scripts/run_e2_optimized_selection_v2.py"),
               "--execute","--engine-root",str(REPO),"--controller-commit",OPTIMIZATION_COMMIT,
               "--output",str(OPTIMIZATION_OUTPUT),"--legacy-cache",str(CAMPAIGN_OUTPUT),
               "--data-root",str(DATA_CACHE),"--source-root-itransformer",str(REPO/"work/author_sources/iTransformer")]
    # Capture stderr explicitly: inherited subprocess streams are not reliably
    # visible in Colab. Keep the redacted local log outside the public campaign.
    import threading, uuid
    diagnostic_directory = WORKSPACE/"diagnostics"
    diagnostic_directory.mkdir(parents=True,exist_ok=True)
    E2_OPT_LOG_PATH = diagnostic_directory/("selection-11-v2-"+uuid.uuid4().hex+".log")
    def redact_e2_diagnostic(text):
        for key in ("GITHUB_TOKEN","GH_TOKEN"):
            secret = GITHUB_ENV.get(key)
            if secret:
                text = text.replace(secret,"[REDACTED]")
        return text
    E2_OPT_ACTIVE_PROCESS = subprocess.Popen(
        command,cwd=OPTIMIZATION_SOURCE,env=GITHUB_ENV,start_new_session=True,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    def stream_e2_output(process,log_path):
        with log_path.open("x",encoding="utf-8") as log:
            for line in process.stdout:
                safe = redact_e2_diagnostic(line)
                log.write(safe)
                log.flush()
                print(safe,end="",flush=True)
    E2_OPT_LOG_THREAD = threading.Thread(
        target=stream_e2_output,args=(E2_OPT_ACTIVE_PROCESS,E2_OPT_LOG_PATH),daemon=True)
    E2_OPT_LOG_THREAD.start()
    print("Controlador optimizado PID:",E2_OPT_ACTIVE_PROCESS.pid,flush=True)
    print("Diagnóstico local:",E2_OPT_LOG_PATH,flush=True)
    interrupted = False
    while True:
        try:
            E2_OPT_RETURN_CODE = E2_OPT_ACTIVE_PROCESS.wait()
            break
        except KeyboardInterrupt:
            if not interrupted:
                interrupted = True
                try:E2_OPT_ACTIVE_PROCESS.send_signal(signal.SIGINT)
                except ProcessLookupError:pass
                print("Interrupción enviada; esperando cierre del trabajador y del diario.",flush=True)
    E2_OPT_LOG_THREAD.join(timeout=10)
    print("Código de salida:",E2_OPT_RETURN_CODE)
    print("Revisa la sección 9. Una pausa o reconciliation_required no equivale a completar E2.")


## 9. Informe de esta campaña

completed_jobs cuenta sólo corridas nuevas; reused_jobs distingue las anteriores efectivamente recorridas. El informe y la rama son los de e2-cuda-crop-11-v2, no el status antiguo de la campaña original.

In [ ]:
print("Código de salida de esta invocación:", globals().get("E2_OPT_RETURN_CODE", "no ejecutada"))
path = OPTIMIZATION_OUTPUT/"status.json"
if path.is_file():
    print(path.read_text())
else:
    print("Sin informe de campaña optimizada: revisa la salida de 8; puede haberse bloqueado antes de sellar.")

log_path = globals().get("E2_OPT_LOG_PATH")
if log_path is not None and log_path.is_file():
    print("\nÚltimas líneas del diagnóstico local:")
    print(redact_e2_diagnostic(log_path.read_text(encoding="utf-8")[-16000:]))


## 10. Diagnóstico si el controlador sale con código 2

Ejecuta esta celda en la misma sesión preparada cuando 8 haya terminado. No repitas 8 para obtener el error. Comprueba fuentes, entorno y perfil CUDA **sintético**, y reproduce las comprobaciones previas al sellado sobre una copia temporal del historial. No abre nuevos CSV de mercado, no inicia entrenamientos/sesiones ni escribe en GitHub; no resuelve ni repite intentos desconocidos. Consume recursos para la comprobación, que no se presentan como corridas científicas. Comparte la salida desde `DIAGNOSTICO` hasta la última línea; los secretos conocidos se ocultan.

Incluye la validación de tipos del constructor real del diario, además de las comprobaciones semánticas del plan.


In [ ]:
# Diagnóstico independiente: no inicia búsqueda ni modifica campañas.
import hashlib, urllib.request, subprocess

active = globals().get("E2_OPT_ACTIVE_PROCESS")
if active is not None and active.poll() is None:
    raise RuntimeError("Hay un controlador activo; espera su cierre.")
required = ("GPU_PYTHON","OPTIMIZATION_SOURCE","OPTIMIZATION_COMMIT","REPO","GITHUB_ENV","WORKSPACE")
if any(name not in globals() for name in required):
    raise RuntimeError("Conserva la sesión preparada de 1–7 y las variables de 8.")

url = "https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/8e3ab6cfb1b3fc3c804f0c914fa5d92b661058d9/scripts/diagnose_e2_optimized_selection.py"
raw = urllib.request.urlopen(url,timeout=30).read()
if hashlib.sha256(raw).hexdigest() != "c90e28f926403014869804cc41cb145e37c4c83da0876325628bf984775f1217":
    raise RuntimeError("El diagnóstico descargado no coincide con la versión fijada.")
diagnostic_script = WORKSPACE/"diagnose_e2_optimized_selection.py"
diagnostic_script.write_bytes(raw)
print("Comprobando fuentes, CUDA, perfil sintético e historial; no se entrenará mercado.",flush=True)
result = subprocess.run(
    [str(GPU_PYTHON),"-u",str(diagnostic_script),
     "--controller-root",str(OPTIMIZATION_SOURCE),
     "--engine-root",str(REPO),"--controller-commit",OPTIMIZATION_COMMIT],
    env=GITHUB_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=600)
diagnostic_text = result.stdout
for key in ("GITHUB_TOKEN","GH_TOKEN"):
    secret = GITHUB_ENV.get(key)
    if secret:
        diagnostic_text = diagnostic_text.replace(secret,"[REDACTED]")
print(diagnostic_text)
print("Código del diagnóstico (no del entrenamiento):",result.returncode)


## 11. Reanudar una pausa por presupuesto con autorización explícita

Sólo después de que 8 termine con `paused_campaign_budget`. Conserva la sesión preparada: no reinicies 1–7 ni repitas intentos desconocidos. Activa `RUN_RESUME_BUDGET=True` una vez. La celda pide un NUEVO LÍMITE TOTAL: por ejemplo, 56 equivale a ocho horas más que el límite inicial de 48. Vacío/0 cancela. Consulta el límite efectivo remoto, conserva consumo/plan/resultados y registra un evento de ampliación confirmado en GitHub antes de reanudar. El mismo límite ya registrado no se suma otra vez. No cambia arquitecturas, orígenes, semillas o topes por ajuste, ni garantiza terminar E2 o prolongar la disponibilidad de Colab.


In [ ]:
RUN_RESUME_BUDGET = False
if RUN_RESUME_BUDGET:
    """Colab entrypoint: ask for explicit total hours, register, then resume saved work."""
    
    
    def resume_e2_after_budget_pause(namespace):
        import hashlib
        import json
        import math
        from pathlib import Path
        import subprocess
        import urllib.request
    
        required = ("WORKSPACE", "GPU_PYTHON", "OPTIMIZATION_SOURCE", "OPTIMIZATION_COMMIT",
                    "REPO", "GITHUB_ENV", "RUN_SEARCH", "RETRY_LOST_ONCE")
        if any(key not in namespace for key in required):
            raise RuntimeError("Faltan variables: usa la misma sesión preparada de Colab, sin reiniciarla.")
        for prefix in ("E2_OPT", "E2_RECOVERY_V4", "E2_RECOVERY_V3", "E2_GUIDE"):
            process = namespace.get(prefix + "_ACTIVE_PROCESS")
            if process is not None and process.poll() is None:
                raise RuntimeError("Hay un controlador activo: espera su cierre.")
        markers = ("run_e2_cuda_search.py", "run_e2_cuda_recovery_", "e2_guide_schedule.py", "run_e2_optimized_selection")
        processes = subprocess.run(["ps", "-eo", "args="], check=True, capture_output=True, text=True).stdout
        if any(any(marker in row for marker in markers) for row in processes.splitlines()):
            raise RuntimeError("Se encontró otro controlador; no se modificó el presupuesto.")
        if not namespace["RUN_SEARCH"] or namespace["RETRY_LOST_ONCE"]:
            raise RuntimeError("Conserva RUN_SEARCH=True y RETRY_LOST_ONCE=False.")
        if not all(namespace.get(key, False) for key in
                ("SOURCE_READY", "PLAN_READY", "REMOTE_READY", "GPU_READY", "SYNTHETIC_READY", "DATA_READY")):
            raise RuntimeError("Completa la preparación 1–7 antes de autorizar una ampliación.")
        if namespace["OPTIMIZATION_COMMIT"] != "8e3ab6cfb1b3fc3c804f0c914fa5d92b661058d9":
            raise RuntimeError("Esta celda requiere el controlador vigente con corrección JSON.")
        print("La campaña pausó por presupuesto. El límite inicial combinado era 48 horas.", flush=True)
        print("Indica un NUEVO LÍMITE TOTAL: por ejemplo, 56 añade 8 horas al inicial; no significa 56 horas adicionales.", flush=True)
        print("Sólo se ampliará el tiempo permitido; modelos, once orígenes, semillas y resultados se conservan.", flush=True)
        answer = input("Nuevo límite TOTAL en horas (vacío o 0 = cancelar): ").strip()
        if not answer:
            print("Cancelado: sin cambios de presupuesto ni entrenamiento.")
            return
        hours = float(answer.replace(",", "."))
        if hours == 0:
            print("Cancelado: sin cambios de presupuesto ni entrenamiento.")
            return
        if not math.isfinite(hours) or hours <= 48:
            raise ValueError("Escribe un límite total mayor que 48, o deja vacío para cancelar.")
        helper_url = ("https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
            "7d0b7be4f9bba068d3fd2d405921d17bb78bb7df/scripts/extend_e2_optimized_budget.py")
        raw = urllib.request.urlopen(helper_url, timeout=30).read()
        if hashlib.sha256(raw).hexdigest() != "1a253aef710edb832a61f2cbc9e14ed659dc2fd9939ebe614b31350ca591be41":
            raise RuntimeError("La descarga no coincide con la versión fijada del ampliador.")
        helper_path = Path(namespace["WORKSPACE"]) / "extend_e2_optimized_budget.py"
        helper_path.write_bytes(raw)
        print("Registrando la ampliación autorizada antes de reanudar...", flush=True)
        result = subprocess.run([str(namespace["GPU_PYTHON"]), "-u", str(helper_path),
            "--controller-root", str(namespace["OPTIMIZATION_SOURCE"]),
            "--engine-root", str(namespace["REPO"]),
            "--controller-commit", namespace["OPTIMIZATION_COMMIT"],
            "--total-hours", str(hours), "--confirmed-by-user"],
            env=namespace["GITHUB_ENV"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, timeout=600)
        output = result.stdout
        for key in ("GITHUB_TOKEN", "GH_TOKEN"):
            secret = namespace["GITHUB_ENV"].get(key)
            if secret:
                output = output.replace(secret, "[REDACTED]")
        print(output, flush=True)
        if result.returncode:
            raise RuntimeError("No se confirmó la ampliación. No se lanzó entrenamiento; conserva este diagnóstico.")
        notebook_url = ("https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
            "91a7bc55e34fab6313e573f7fe11dda15706982a/notebooks/entregable2_optimizacion_11_origenes.ipynb")
        notebook_raw = urllib.request.urlopen(notebook_url, timeout=30).read()
        if hashlib.sha256(notebook_raw).hexdigest() != "43b45a6405a696341c8727384904de42ca7250c529a92766320f094cd512278d":
            raise RuntimeError("El notebook descargado no coincide con la versión fijada.")
        notebook = json.loads(notebook_raw)
        code = "".join(notebook["cells"][17]["source"])
        if not code.startswith("RUN_OPTIMIZED_SELECTION = False"):
            raise RuntimeError("La sección de reanudación no coincide con la versión fijada.")
        code = code.replace("RUN_OPTIMIZED_SELECTION = False", "RUN_OPTIMIZED_SELECTION = True", 1)
        print("Ampliación confirmada. Reanudando desde las corridas guardadas...", flush=True)
        exec(compile(code, "seccion_8_reanudada_con_presupuesto_registrado", "exec"), namespace)
    
    
    resume_e2_after_budget_pause(globals())
    
else:
    print('Reanudación manual desactivada; no se cambió el presupuesto.')


## 12. Supervisar sin ampliar las 56 horas autorizadas

Opcional: activa RUN_SUPERVISE_56=True y ejecuta esta celda después de la invocación actual; puede dejarse en cola debajo de ella. Espera al controlador conocido sin interrumpirlo. Sólo reanuda pausas limpias por presupuesto de **sesión**, conservando el saldo acumulado autorizado y los resultados. Se detiene por presupuesto global, falta de reserva, error, conciliación o ausencia de progreso. No añade horas, no cambia el ejecutor científico y no mantiene viva ni reconecta una VM cerrada por Colab. Las 56 horas son acumuladas, no 56 adicionales.


In [ ]:
RUN_SUPERVISE_56 = False
if RUN_SUPERVISE_56:
    """Resume only clean session-budget pauses; never extend the authorized 56-hour total."""
    
    import hashlib
    import json
    import math
    from pathlib import Path
    
    MAX_TOTAL_SECONDS = 56 * 3600.0
    CONTROLLER_COMMIT = "8e3ab6cfb1b3fc3c804f0c914fa5d92b661058d9"
    BRANCH = "campaigns/e2-cuda-crop-11-v2"
    
    
    def checked_report(output):
        output = Path(output)
        config = json.loads((output / "campaign.json").read_text())
        status = json.loads((output / "status.json").read_text())
        plan = config["plan"]
        canonical = json.dumps(plan, sort_keys=True, separators=(",", ":"),
                               ensure_ascii=False, allow_nan=False).encode()
        digest = hashlib.sha256(canonical).hexdigest()
        if config["plan_sha256"] != digest or status["plan_sha256"] != digest:
            raise ValueError("campaign_or_report_plan_mismatch")
        if status["remote_branch"] != BRANCH or plan["remote_storage"]["branch"] != BRANCH:
            raise ValueError("unexpected_campaign_branch")
        budget = status["budget"]
        if budget["plan_sha256"] != digest:
            raise ValueError("budget_plan_mismatch")
        values = [plan["legacy_cache"]["spent_seconds"], budget["campaign_budget_seconds"],
                  budget["charged_wall_seconds"], budget["remaining_campaign_seconds"],
                  status["combined_charged_seconds"]]
        if any(type(x) not in (int, float) or not math.isfinite(x) or x < 0 for x in values):
            raise ValueError("invalid_budget_metadata")
        legacy, limit, charged, remaining, combined = values
        if (not math.isclose(combined, legacy + charged, rel_tol=0, abs_tol=1e-5)
                or not math.isclose(remaining, max(0., limit-charged), rel_tol=0, abs_tol=1e-5)):
            raise ValueError("inconsistent_budget_metadata")
        count = status["completed_jobs"]
        if (type(count) is not int or type(budget["completed_jobs"]) is not int
                or count < 0 or budget["completed_jobs"] != count):
            raise ValueError("invalid_completed_result_counter")
        return status, legacy + limit
    
    
    def supervise(namespace, invoke):
        """Wait for the current invocation, then resume session pauses one at a time.
    
        Existing immutable records and budget enforcement are owned by the same
        frozen controller. No budget extension, signal, kill or error retry occurs.
        """
        previous_count = None
        resumptions = 0
        while True:
            process = namespace.get("E2_OPT_ACTIVE_PROCESS")
            if process is None:
                raise RuntimeError("No hay un controlador conocido; conserva la sesión preparada.")
            if process.poll() is None:
                print("Supervisor: esperando el cierre normal del controlador actual, sin interrumpirlo.", flush=True)
                return_code = process.wait()
            else:
                return_code = process.poll()
            if return_code != 0:
                print("Supervisor detenido: el controlador salió con código", return_code, flush=True)
                return {"stop_reason": "controller_nonzero_exit", "return_code": return_code,
                        "automatic_resumptions": resumptions}
            status, total = checked_report(namespace["OPTIMIZATION_OUTPUT"])
            if total > MAX_TOTAL_SECONDS + 1e-5:
                raise ValueError("combined_budget_exceeds_authorized_56_hours")
            state = status["status"]
            if state != "paused_session_budget":
                print("Supervisor detenido con estado:", state, flush=True)
                if state == "paused_campaign_budget":
                    print("No se amplía el límite. El saldo puede ser insuficiente para reservar otro ajuste.", flush=True)
                return {"stop_reason": state, "completed_jobs": status["completed_jobs"],
                        "automatic_resumptions": resumptions}
            if status.get("pause_reason") != "session_budget":
                raise ValueError("session_pause_reason_mismatch")
            count = status["completed_jobs"]
            if previous_count is not None and count <= previous_count:
                raise RuntimeError("session_resume_made_no_new_progress; automatic_retry_stopped")
            if status["budget"]["remaining_campaign_seconds"] <= 0:
                print("Supervisor detenido: sin saldo de campaña.", flush=True)
                return {"stop_reason": "no_remaining_campaign_budget", "completed_jobs": count,
                        "automatic_resumptions": resumptions}
            previous_count = count
            resumptions += 1
            print("Supervisor: reanudación automática", resumptions,
                  "tras pausa de sesión; límite acumulado", total/3600,
                  "horas; resultados conservados", count, flush=True)
            invoke(namespace)
    
    
    def run_prepared_colab(namespace):
        import urllib.request
        if namespace.get("OPTIMIZATION_COMMIT") != CONTROLLER_COMMIT:
            raise RuntimeError("Se requiere el controlador vigente y la misma sesión preparada.")
        required = ("OPTIMIZATION_OUTPUT", "E2_OPT_ACTIVE_PROCESS", "GPU_PYTHON", "GITHUB_ENV")
        if any(name not in namespace for name in required):
            raise RuntimeError("Faltan variables del entrenamiento; no reinicies el kernel.")
        url = ("https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
               "91a7bc55e34fab6313e573f7fe11dda15706982a/"
               "notebooks/entregable2_optimizacion_11_origenes.ipynb")
        raw = urllib.request.urlopen(url, timeout=30).read()
        if hashlib.sha256(raw).hexdigest() != "43b45a6405a696341c8727384904de42ca7250c529a92766320f094cd512278d":
            raise RuntimeError("notebook_source_hash_mismatch")
        code = "".join(json.loads(raw)["cells"][17]["source"])
        if not code.startswith("RUN_OPTIMIZED_SELECTION = False"):
            raise RuntimeError("unexpected_frozen_controller_cell")
        code = code.replace("RUN_OPTIMIZED_SELECTION = False", "RUN_OPTIMIZED_SELECTION = True", 1)
        compiled = compile(code, "seccion8_reanudacion_automatica_sin_ampliar", "exec")
        print("Supervisor: sólo reanuda pausas de sesión; no amplía las 56 horas ni evita cierres de Colab.", flush=True)
        result = supervise(namespace, lambda ns: exec(compiled, ns))
        print(json.dumps(result, ensure_ascii=False), flush=True)
        return result
    
    
    if __name__ == "__main__":
        run_prepared_colab(globals())
    
else:
    print('Supervisor desactivado; no se inició otro controlador ni se amplió presupuesto.')
